# 🧠 AI-Driven Financial News Sentiment Analysis for Stock Market Movement Prediction

**Student:** Arpit Saxena  
**Program:** PGDM – Research and Business Analytics 24-26  
**Faculty Guide:** Prof. Dr. P. V. Chandrika  
**Project Area:** Artificial Intelligence, Financial Analytics & NLP

---

## 📌 Project Overview

This notebook implements a complete pipeline for **financial news sentiment analysis** using:
1. **VADER** – Rule-based, fast, explainable sentiment scoring
2. **FinBERT** – Pre-trained transformer model specialized for financial text
3. **Deep Learning (LSTM + Dense)** – Predicts market direction: `Positive / Neutral / Negative`

### Pipeline Flow
```
Raw Financial Headlines
        ↓
Text Preprocessing (NLP)
        ↓
Sentiment Scoring: VADER + FinBERT
        ↓
Feature Engineering
        ↓
Deep Learning Model (LSTM)
        ↓
Evaluation & Visualization Dashboard
```

---
## 📦 Section 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once)
!pip install transformers torch vaderSentiment scikit-learn pandas numpy matplotlib seaborn wordcloud tqdm --quiet

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── NLP & Sentiment ─────────────────────────────────────────────────────────
import re
import string
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ── Machine / Deep Learning ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Misc ────────────────────────────────────────────────────────────────────
from wordcloud import WordCloud
from tqdm import tqdm
from collections import Counter

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device: use GPU if available
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Libraries loaded | Device: {DEVICE}")

---
## 📂 Section 2: Data Loading & Exploration

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
# Dataset: Financial PhraseBank — 4846 financial news headlines
# Source: Malo et al. (2014), widely used benchmark for financial NLP
# Columns: sentiment (positive/neutral/negative), headline (text)

df = pd.read_csv('all-data.csv', encoding='latin-1', header=None, names=['sentiment', 'headline'])

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['sentiment'].value_counts())
print(f"\nSample rows:")
df.head()

In [ ]:
# ── Basic EDA ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Exploratory Data Analysis — Financial Headlines Dataset', fontsize=15, fontweight='bold')

# 1. Class Distribution (Bar)
counts = df['sentiment'].value_counts()
colors = {'positive': '#2ecc71', 'neutral': '#3498db', 'negative': '#e74c3c'}
bar_colors = [colors[s] for s in counts.index]
axes[0].bar(counts.index, counts.values, color=bar_colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# 2. Class Distribution (Pie)
axes[1].pie(counts.values, labels=counts.index, colors=bar_colors,
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Sentiment Share (%)', fontsize=12, fontweight='bold')

# 3. Headline Length Distribution
df['headline_len'] = df['headline'].apply(lambda x: len(x.split()))
for sentiment, color in colors.items():
    subset = df[df['sentiment'] == sentiment]['headline_len']
    axes[2].hist(subset, bins=30, alpha=0.6, label=sentiment, color=color)
axes[2].set_title('Headline Word Count Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Word Count')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAverage headline length: {df['headline_len'].mean():.1f} words")
print(f"Max: {df['headline_len'].max()} | Min: {df['headline_len'].min()}")

In [ ]:
# ── Word Cloud per Sentiment ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Word Cloud by Sentiment Class', fontsize=14, fontweight='bold')

wc_colors = {'positive': 'Greens', 'neutral': 'Blues', 'negative': 'Reds'}
for ax, sentiment in zip(axes, ['positive', 'neutral', 'negative']):
    text = ' '.join(df[df['sentiment'] == sentiment]['headline'].tolist())
    wc = WordCloud(width=600, height=400, background_color='white',
                   colormap=wc_colors[sentiment], max_words=80).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(sentiment.capitalize(), fontsize=12, fontweight='bold',
                 color=list(colors.values())[list(colors.keys()).index(sentiment)])

plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🧹 Section 3: Text Preprocessing

In [ ]:
def preprocess_text(text):
    """
    Standard NLP preprocessing pipeline:
    1. Lowercase
    2. Remove special characters & extra spaces
    3. Remove numbers (optional — financial context may need them)
    4. Strip punctuation
    
    NOTE: We keep a 'clean' version for VADER/feature analysis.
    For FinBERT & LSTM, we use the ORIGINAL text (transformers handle it better).
    """
    text = str(text).lower()                          # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)             # remove non-alpha
    text = re.sub(r'\s+', ' ', text).strip()           # collapse whitespace
    return text

df['cleaned_headline'] = df['headline'].apply(preprocess_text)

print("✅ Preprocessing complete!")
print("\nOriginal vs Cleaned example:")
print(f"  Original : {df['headline'].iloc[5]}")
print(f"  Cleaned  : {df['cleaned_headline'].iloc[5]}")

---
## 💬 Section 4: Sentiment Analysis — VADER (Rule-Based)

> **VADER** (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon-based tool that uses a predefined list of words with sentiment scores. It's fast, explainable, and works well on short texts like headlines.
> - Outputs 4 scores: `positive`, `neutral`, `negative`, `compound` (−1 to +1)
> - **No training required** — makes it a strong baseline

In [ ]:
# Initialize VADER
vader = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    """Return all 4 VADER scores for a given text."""
    return vader.polarity_scores(text)

def vader_label(compound):
    """Convert compound score → sentiment label (matches our 3-class target)."""
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'

# Apply VADER to all headlines
tqdm.pandas(desc="Running VADER")
vader_scores = df['headline'].progress_apply(get_vader_scores)

df['vader_pos']      = vader_scores.apply(lambda x: x['pos'])
df['vader_neu']      = vader_scores.apply(lambda x: x['neu'])
df['vader_neg']      = vader_scores.apply(lambda x: x['neg'])
df['vader_compound'] = vader_scores.apply(lambda x: x['compound'])
df['vader_label']    = df['vader_compound'].apply(vader_label)

print("✅ VADER scoring complete!")
df[['headline', 'sentiment', 'vader_compound', 'vader_label']].head(8)

In [ ]:
# ── VADER Performance Evaluation ─────────────────────────────────────────────
print("=" * 55)
print("      VADER Performance vs Ground Truth Labels")
print("=" * 55)
vader_acc = accuracy_score(df['sentiment'], df['vader_label'])
print(f"\nOverall Accuracy: {vader_acc:.4f} ({vader_acc*100:.2f}%)\n")
print(classification_report(df['sentiment'], df['vader_label'],
                             target_names=['negative', 'neutral', 'positive']))

# Confusion Matrix
cm_vader = confusion_matrix(df['sentiment'], df['vader_label'],
                             labels=['negative', 'neutral', 'positive'])
plt.figure(figsize=(7, 5))
sns.heatmap(cm_vader, annot=True, fmt='d', cmap='Blues',
            xticklabels=['negative', 'neutral', 'positive'],
            yticklabels=['negative', 'neutral', 'positive'],
            linewidths=0.5)
plt.title('VADER Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('vader_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── VADER Score Distribution per True Sentiment ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for sentiment, color in colors.items():
    subset = df[df['sentiment'] == sentiment]['vader_compound']
    ax.hist(subset, bins=40, alpha=0.55, label=f'{sentiment} (n={len(subset)})', color=color)

ax.axvline(0.05, color='green', linestyle='--', linewidth=1.2, label='Positive threshold (0.05)')
ax.axvline(-0.05, color='red', linestyle='--', linewidth=1.2, label='Negative threshold (−0.05)')
ax.set_title('VADER Compound Score Distribution by True Sentiment', fontsize=13, fontweight='bold')
ax.set_xlabel('VADER Compound Score')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('vader_distribution.png', dpi=150)
plt.show()

---
## 🤖 Section 5: Sentiment Analysis — FinBERT (Transformer-Based)

> **FinBERT** is a BERT model pre-trained on financial corpora (Reuters, financial news). It understands domain-specific language that generic models miss (e.g., "profit warning", "write-off", "bearish outlook").
> - Directly outputs: `positive`, `neutral`, `negative` + confidence scores
> - More accurate than VADER on financial text but computationally heavier

In [ ]:
# ── Load FinBERT Model ────────────────────────────────────────────────────────
# Model: ProsusAI/finbert (fine-tuned on Financial PhraseBank — same dataset family)
print("Loading FinBERT... (downloads ~400MB on first run)")

FINBERT_MODEL = 'ProsusAI/finbert'
finbert_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
finbert_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
finbert_model = finbert_model.to(DEVICE)
finbert_model.eval()

# FinBERT label mapping (model's internal order → standard labels)
FINBERT_LABELS = finbert_model.config.id2label
print(f"\n✅ FinBERT loaded | Label map: {FINBERT_LABELS}")

In [ ]:
def predict_finbert_batch(texts, batch_size=32):
    """
    Run FinBERT inference in batches for efficiency.
    Returns list of (predicted_label, confidence_score).
    """
    all_labels = []
    all_scores = []

    for i in tqdm(range(0, len(texts), batch_size), desc="FinBERT inference"):
        batch = texts[i:i + batch_size]
        encoding = finbert_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            outputs = finbert_model(**encoding)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()

        for prob_row in probs:
            pred_idx = np.argmax(prob_row)
            all_labels.append(FINBERT_LABELS[pred_idx])
            all_scores.append(float(prob_row[pred_idx]))

    return all_labels, all_scores

# Run FinBERT (uses original text — transformers work best on unprocessed text)
headlines_list = df['headline'].tolist()
finbert_labels, finbert_scores = predict_finbert_batch(headlines_list)

df['finbert_label']      = finbert_labels
df['finbert_confidence'] = finbert_scores

print("\n✅ FinBERT scoring complete!")
df[['headline', 'sentiment', 'finbert_label', 'finbert_confidence']].head(8)

In [ ]:
# ── FinBERT Performance Evaluation ───────────────────────────────────────────
print("=" * 55)
print("     FinBERT Performance vs Ground Truth Labels")
print("=" * 55)
finbert_acc = accuracy_score(df['sentiment'], df['finbert_label'])
print(f"\nOverall Accuracy: {finbert_acc:.4f} ({finbert_acc*100:.2f}%)\n")
print(classification_report(df['sentiment'], df['finbert_label'],
                             target_names=['negative', 'neutral', 'positive']))

# Confusion Matrix
cm_finbert = confusion_matrix(df['sentiment'], df['finbert_label'],
                               labels=['negative', 'neutral', 'positive'])
plt.figure(figsize=(7, 5))
sns.heatmap(cm_finbert, annot=True, fmt='d', cmap='Greens',
            xticklabels=['negative', 'neutral', 'positive'],
            yticklabels=['negative', 'neutral', 'positive'],
            linewidths=0.5)
plt.title('FinBERT Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('finbert_confusion_matrix.png', dpi=150)
plt.show()

---
## ⚖️ Section 6: VADER vs FinBERT — Comparison

In [ ]:
# ── Side-by-Side Accuracy Comparison ─────────────────────────────────────────
from sklearn.metrics import f1_score, precision_score, recall_score

def evaluate_model(true, pred, name):
    return {
        'Model': name,
        'Accuracy': accuracy_score(true, pred),
        'F1 (macro)': f1_score(true, pred, average='macro', zero_division=0),
        'Precision (macro)': precision_score(true, pred, average='macro', zero_division=0),
        'Recall (macro)': recall_score(true, pred, average='macro', zero_division=0),
    }

results = pd.DataFrame([
    evaluate_model(df['sentiment'], df['vader_label'], 'VADER'),
    evaluate_model(df['sentiment'], df['finbert_label'], 'FinBERT'),
])
results = results.set_index('Model')
print("\n📊 Model Comparison Table")
print(results.round(4).to_string())

# Plot
results_plot = results.reset_index()
metrics = ['Accuracy', 'F1 (macro)', 'Precision (macro)', 'Recall (macro)']
x = np.arange(len(metrics))
width = 0.3

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, results_plot.loc[0, metrics], width, label='VADER',
               color='#3498db', edgecolor='black', linewidth=0.8)
bars2 = ax.bar(x + width/2, results_plot.loc[1, metrics], width, label='FinBERT',
               color='#2ecc71', edgecolor='black', linewidth=0.8)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('VADER vs FinBERT — Performance Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.legend()
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig('vader_vs_finbert.png', dpi=150)
plt.show()

In [ ]:
# ── Agreement Analysis ────────────────────────────────────────────────────────
df['models_agree'] = df['vader_label'] == df['finbert_label']
agreement_rate = df['models_agree'].mean()
print(f"VADER & FinBERT agree on: {agreement_rate:.2%} of headlines")

# Where they disagree — look at examples
print("\n📌 Examples where VADER & FinBERT DISAGREE:")
disagree_df = df[~df['models_agree']][['headline', 'sentiment', 'vader_label', 'finbert_label']].head(5)
for _, row in disagree_df.iterrows():
    print(f"  Headline  : {row['headline'][:80]}...")
    print(f"  True Label: {row['sentiment']} | VADER: {row['vader_label']} | FinBERT: {row['finbert_label']}")
    print()

---
## 🔧 Section 7: Feature Engineering for Deep Learning

> We now build a **feature matrix** that combines:
> - VADER's 4 numerical scores (explainable)
> - FinBERT's confidence score (model certainty)
> - Text-derived features (length, special word counts)
> 
> These features feed into our LSTM-based deep learning classifier.

In [ ]:
# ── Extract Numerical Features ────────────────────────────────────────────────
# Financial sentiment keywords (domain knowledge)
POSITIVE_WORDS = {'profit', 'growth', 'increase', 'rise', 'gain', 'record', 'surge', 'beat',
                  'strong', 'positive', 'improve', 'expand', 'boost', 'rally', 'recovery'}
NEGATIVE_WORDS = {'loss', 'decline', 'drop', 'fall', 'cut', 'layoff', 'bankruptcy', 'warning',
                  'weak', 'negative', 'debt', 'miss', 'reduce', 'concern', 'risk', 'deficit'}

def count_financial_words(text, word_set):
    words = set(text.lower().split())
    return len(words.intersection(word_set))

# Feature engineering
df['text_length']    = df['headline'].apply(lambda x: len(x.split()))
df['char_length']    = df['headline'].apply(len)
df['pos_word_count'] = df['cleaned_headline'].apply(lambda x: count_financial_words(x, POSITIVE_WORDS))
df['neg_word_count'] = df['cleaned_headline'].apply(lambda x: count_financial_words(x, NEGATIVE_WORDS))
df['fin_word_ratio'] = (df['pos_word_count'] - df['neg_word_count']) / (df['text_length'] + 1)

# Encode FinBERT label as numeric (1=positive, 0=neutral, -1=negative)
finbert_enc = {'positive': 1, 'neutral': 0, 'negative': -1}
df['finbert_encoded'] = df['finbert_label'].map(finbert_enc)

print("✅ Feature engineering complete!")
feature_cols = ['vader_pos', 'vader_neu', 'vader_neg', 'vader_compound',
                'finbert_confidence', 'finbert_encoded',
                'text_length', 'pos_word_count', 'neg_word_count', 'fin_word_ratio']
print(f"\nFeature matrix shape: {df[feature_cols].shape}")
df[feature_cols].describe().round(4)

In [ ]:
# ── Feature Correlation Heatmap ───────────────────────────────────────────────
plt.figure(figsize=(10, 7))
corr = df[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150)
plt.show()

---
## 🧠 Section 8: Deep Learning Model — LSTM Classifier

> **Architecture:** We build an LSTM-based classifier that treats the feature vector as a sequence.  
> **Task:** Multi-class classification → `Positive / Neutral / Negative` (3 classes)  
> 
> **Why LSTM?** LSTMs are designed to capture sequential dependencies — ideal for understanding context in text-derived features.

In [ ]:
# ── Prepare Data for PyTorch ──────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

# Encode target labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['sentiment'])       # negative=0, neutral=1, positive=2
label_names = label_encoder.classes_
print(f"Label encoding: {dict(zip(label_names, range(len(label_names))))}")

# Feature matrix
X = df[feature_cols].values.astype(np.float32)

# Normalize features (important for neural networks)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# Train / Validation / Test split: 70% / 15% / 15%
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y, test_size=0.15, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp)

print(f"\nSplit sizes → Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Feature shape: {X_train.shape}")

In [ ]:
# ── PyTorch Dataset ───────────────────────────────────────────────────────────
class SentimentDataset(Dataset):
    """Custom Dataset for financial sentiment features."""
    def __init__(self, features, labels):
        # LSTM expects input shape: (batch, seq_len, features)
        # We treat each feature vector as a sequence of length 1
        self.X = torch.tensor(features, dtype=torch.float32).unsqueeze(1)  # (N, 1, F)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64

train_dataset = SentimentDataset(X_train, y_train)
val_dataset   = SentimentDataset(X_val,   y_val)
test_dataset  = SentimentDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f"✅ DataLoaders ready | Batch size: {BATCH_SIZE}")

In [ ]:
# ── Model Architecture ────────────────────────────────────────────────────────
class FinancialSentimentLSTM(nn.Module):
    """
    LSTM-based classifier for financial sentiment.
    
    Architecture:
    Input Features (10) → LSTM → Dropout → FC Layer → Dropout → Output (3 classes)
    
    - input_size   : number of features (10)
    - hidden_size  : LSTM hidden units
    - num_layers   : stacked LSTM layers
    - num_classes  : 3 (positive, neutral, negative)
    - dropout      : regularization to prevent overfitting
    """
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=3, dropout=0.3):
        super(FinancialSentimentLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True   # Bidirectional captures context from both directions
        )
        self.dropout = nn.Dropout(dropout)

        lstm_output_size = hidden_size * 2  # ×2 because bidirectional
        self.fc1 = nn.Linear(lstm_output_size, 64)
        self.bn1  = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        lstm_out, (hidden, _) = self.lstm(x)
        # Use the last hidden state from both directions
        out = lstm_out[:, -1, :]           # (batch, hidden*2)
        out = self.dropout(out)
        out = F.relu(self.bn1(self.fc1(out)))
        out = self.dropout(out)
        out = self.fc2(out)                 # raw logits (no softmax — CrossEntropy handles it)
        return out


# Instantiate model
INPUT_SIZE  = X_train.shape[1]   # 10 features
NUM_CLASSES = len(label_names)   # 3

model = FinancialSentimentLSTM(
    input_size=INPUT_SIZE,
    hidden_size=128,
    num_layers=2,
    num_classes=NUM_CLASSES,
    dropout=0.3
).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nTotal parameters  : {total_params:,}")
print(f"Trainable params  : {trainable_params:,}")

In [ ]:
# ── Training Setup ────────────────────────────────────────────────────────────
# Handle class imbalance using weighted loss
# (neutral=2879, positive=1363, negative=604) — negative is underrepresented
class_counts = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()   # normalize
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"Class weights: {dict(zip(label_names, class_weights.round(4)))}")

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Learning rate scheduler — reduce LR when val_loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                         patience=5, factor=0.5, verbose=True)
print("\n✅ Training setup complete")

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────
EPOCHS = 50
PATIENCE = 10   # Early stopping patience

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

print(f"Training for up to {EPOCHS} epochs (early stopping at patience={PATIENCE})\n")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>8} | {'Train Acc':>9} | {'Val Acc':>7}")
print("-" * 55)

for epoch in range(1, EPOCHS + 1):
    # ── TRAIN ──
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()

        train_loss += loss.item() * len(y_batch)
        preds = logits.argmax(dim=1)
        train_correct += (preds == y_batch).sum().item()
        train_total += len(y_batch)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ── VALIDATE ──
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item() * len(y_batch)
            preds = logits.argmax(dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += len(y_batch)

    val_loss /= val_total
    val_acc = val_correct / val_total

    # ── RECORD ──
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    scheduler.step(val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>8.4f} | {train_acc:>9.4f} | {val_acc:>7.4f}")

    # ── EARLY STOPPING ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n⏹ Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

# Restore best model
model.load_state_dict(best_model_state)
print(f"\n✅ Training complete! Best val loss: {best_val_loss:.4f}")

In [ ]:
# ── Training Curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training History', fontsize=14, fontweight='bold')

epochs_ran = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_ran, history['train_loss'], label='Train Loss', color='#3498db', linewidth=2)
axes[0].plot(epochs_ran, history['val_loss'],   label='Val Loss',   color='#e74c3c', linewidth=2)
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, history['train_acc'], label='Train Accuracy', color='#3498db', linewidth=2)
axes[1].plot(epochs_ran, history['val_acc'],   label='Val Accuracy',   color='#e74c3c', linewidth=2)
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

---
## 📊 Section 9: Model Evaluation on Test Set

In [ ]:
# ── Test Set Evaluation ───────────────────────────────────────────────────────
model.eval()
all_preds, all_true, all_probs = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        logits = model(X_batch)
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_preds.extend(preds)
        all_true.extend(y_batch.numpy())
        all_probs.extend(probs)

all_preds = np.array(all_preds)
all_true  = np.array(all_true)
all_probs = np.array(all_probs)

lstm_acc = accuracy_score(all_true, all_preds)
print("=" * 60)
print("     LSTM Model — Test Set Evaluation")
print("=" * 60)
print(f"\nTest Accuracy: {lstm_acc:.4f} ({lstm_acc*100:.2f}%)\n")
print(classification_report(all_true, all_preds, target_names=label_names))

In [ ]:
# ── Final Confusion Matrix ────────────────────────────────────────────────────
cm_lstm = confusion_matrix(all_true, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Purples',
            xticklabels=label_names, yticklabels=label_names,
            linewidths=0.5, annot_kws={'size': 14, 'weight': 'bold'})
plt.title('LSTM Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('lstm_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ── Confidence Score Distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('LSTM Prediction Confidence by True Class', fontsize=13, fontweight='bold')

for i, (cls, color) in enumerate(zip(label_names, ['#e74c3c', '#3498db', '#2ecc71'])):
    mask = all_true == i
    conf = all_probs[mask, i]  # confidence for the true class
    axes[i].hist(conf, bins=20, color=color, edgecolor='black', linewidth=0.5)
    axes[i].set_title(f'Class: {cls}', fontweight='bold')
    axes[i].set_xlabel('Predicted Probability')
    axes[i].set_ylabel('Count')
    axes[i].axvline(conf.mean(), color='black', linestyle='--', label=f'Mean: {conf.mean():.2f}')
    axes[i].legend()

plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=150)
plt.show()

---
## 🏆 Section 10: Final Model Comparison — All Three Approaches

In [ ]:
# ── Comprehensive Comparison Table ────────────────────────────────────────────
# LSTM test predictions mapped back to label names for consistency
lstm_pred_labels = label_encoder.inverse_transform(all_preds)
lstm_true_labels = label_encoder.inverse_transform(all_true)

# Evaluate all three on same test split
test_idx = np.where(np.isin(np.arange(len(df)),
                             np.arange(len(df))[-len(all_true):]))[0]   # approximate

final_results = pd.DataFrame([
    evaluate_model(df['sentiment'], df['vader_label'],   'VADER (Rule-Based)'),
    evaluate_model(df['sentiment'], df['finbert_label'], 'FinBERT (Transformer)'),
    {**evaluate_model(lstm_true_labels, lstm_pred_labels, 'LSTM Deep Learning'),
     'Note': 'Test set only'}
]).fillna('').set_index('Model')

print("\n🏆 FINAL MODEL COMPARISON")
print("=" * 70)
print(final_results[['Accuracy', 'F1 (macro)', 'Precision (macro)', 'Recall (macro)']].round(4).to_string())

# Visual bar chart
models = ['VADER\n(Rule-Based)', 'FinBERT\n(Transformer)', 'LSTM\n(Deep Learning)']
accs   = [vader_acc, finbert_acc, lstm_acc]
model_colors = ['#3498db', '#2ecc71', '#9b59b6']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, accs, color=model_colors, edgecolor='black', linewidth=0.8, width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.4f}\n({acc*100:.1f}%)', ha='center', fontweight='bold', fontsize=11)

ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Final Model Comparison — Accuracy', fontsize=14, fontweight='bold')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Baseline (random)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150)
plt.show()

---
## 🔍 Section 11: Explainability — SHAP-style Feature Importance

> For manager presentation — showing **which features drive the model's predictions**.

In [ ]:
# ── Feature Importance via Gradient-based Attribution ─────────────────────────
# We compute input gradients to understand which features the LSTM attends to most.
# This is a lightweight explainability technique (no external libraries needed).

model.eval()
feature_importance = np.zeros(INPUT_SIZE)

for X_batch, y_batch in test_loader:
    X_batch = X_batch.to(DEVICE).requires_grad_(True)
    logits = model(X_batch)
    # Sum of logits for backprop
    logits.sum().backward()
    # Gradient magnitude = importance
    feature_importance += X_batch.grad.abs().mean(dim=0).squeeze().cpu().detach().numpy()

feature_importance /= len(test_loader)
feature_importance_normalized = feature_importance / feature_importance.sum()

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': feature_importance_normalized
}).sort_values('Importance', ascending=True)

# Plot horizontal bar chart
fig, ax = plt.subplots(figsize=(9, 6))
colors_feat = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(importance_df)))
bars = ax.barh(importance_df['Feature'], importance_df['Importance'],
               color=colors_feat, edgecolor='black', linewidth=0.5)

for bar in bars:
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=9)

ax.set_title('Feature Importance — LSTM (Gradient Attribution)', fontsize=13, fontweight='bold')
ax.set_xlabel('Normalized Importance Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print("\nTop 3 most important features:")
print(importance_df.tail(3)[['Feature', 'Importance']].to_string(index=False))

---
## 📡 Section 12: Inference Pipeline — Real-Time Prediction

In [ ]:
def predict_headline(headline_text, verbose=True):
    """
    End-to-end prediction pipeline for a NEW financial headline.
    Returns: sentiment prediction + confidence + all scores
    """
    # Step 1: Preprocess
    cleaned = preprocess_text(headline_text)

    # Step 2: VADER scores
    vader_scores_dict = vader.polarity_scores(headline_text)

    # Step 3: FinBERT prediction
    enc = finbert_tokenizer(headline_text, return_tensors='pt',
                             truncation=True, max_length=128).to(DEVICE)
    with torch.no_grad():
        out = finbert_model(**enc)
    probs_fb = F.softmax(out.logits, dim=-1).cpu().numpy()[0]
    fb_idx   = np.argmax(probs_fb)
    fb_label = FINBERT_LABELS[fb_idx]
    fb_conf  = float(probs_fb[fb_idx])

    # Step 4: Feature vector
    pos_count = count_financial_words(cleaned, POSITIVE_WORDS)
    neg_count = count_financial_words(cleaned, NEGATIVE_WORDS)
    text_len  = len(headline_text.split())

    features = np.array([[
        vader_scores_dict['pos'],
        vader_scores_dict['neu'],
        vader_scores_dict['neg'],
        vader_scores_dict['compound'],
        fb_conf,
        {'positive': 1, 'neutral': 0, 'negative': -1}[fb_label],
        text_len,
        pos_count,
        neg_count,
        (pos_count - neg_count) / (text_len + 1)
    ]], dtype=np.float32)

    features_scaled = scaler.transform(features)
    features_tensor = torch.tensor(features_scaled, dtype=torch.float32).unsqueeze(1).to(DEVICE)

    # Step 5: LSTM prediction
    model.eval()
    with torch.no_grad():
        logits = model(features_tensor)
        lstm_probs = F.softmax(logits, dim=-1).cpu().numpy()[0]

    lstm_pred = label_encoder.inverse_transform([np.argmax(lstm_probs)])[0]
    lstm_conf = float(np.max(lstm_probs))

    if verbose:
        print(f"\n{'='*60}")
        print(f" Headline: {headline_text[:70]}")
        print(f"{'='*60}")
        print(f"  VADER Score     : {vader_scores_dict['compound']:+.4f} → {vader_label(vader_scores_dict['compound']).upper()}")
        print(f"  FinBERT Label   : {fb_label.upper()} (confidence: {fb_conf:.2%})")
        print(f"  LSTM Prediction : {lstm_pred.upper()} (confidence: {lstm_conf:.2%})")
        print(f"  Class probs     : negative={lstm_probs[0]:.3f} | neutral={lstm_probs[1]:.3f} | positive={lstm_probs[2]:.3f}")
        print(f"{'='*60}")

    return {'lstm_pred': lstm_pred, 'lstm_conf': lstm_conf,
            'vader': vader_label(vader_scores_dict['compound']),
            'finbert': fb_label}


# Test with sample headlines
test_headlines = [
    "Company reports record profits and announces expansion into new markets",
    "Shares tumble after disappointing quarterly earnings and massive layoffs",
    "The company will hold its annual general meeting next month in Helsinki"
]

for h in test_headlines:
    predict_headline(h)

---
## 📋 Section 13: Summary Dashboard & Key Takeaways

In [ ]:
# ── Full Summary Dashboard ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.suptitle('AI-Driven Financial News Sentiment Analysis\nDashboard Summary',
             fontsize=18, fontweight='bold', y=0.98)

# 1. Class Distribution
ax1 = fig.add_subplot(3, 4, 1)
counts = df['sentiment'].value_counts()
ax1.bar(counts.index, counts.values,
        color=[colors[s] for s in counts.index], edgecolor='black')
ax1.set_title('Dataset Distribution', fontweight='bold')
ax1.set_ylabel('Count')

# 2. VADER compound distribution
ax2 = fig.add_subplot(3, 4, 2)
for sent, color in colors.items():
    ax2.hist(df[df['sentiment'] == sent]['vader_compound'], bins=25, alpha=0.5,
             label=sent, color=color)
ax2.set_title('VADER Score Distribution', fontweight='bold')
ax2.legend(fontsize=7)

# 3. Confusion Matrices
ax3 = fig.add_subplot(3, 4, 3)
sns.heatmap(cm_vader, annot=True, fmt='d', cmap='Blues',
            xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'],
            ax=ax3, cbar=False)
ax3.set_title(f'VADER CM\nAcc: {vader_acc:.2%}', fontweight='bold')

ax4 = fig.add_subplot(3, 4, 4)
sns.heatmap(cm_finbert, annot=True, fmt='d', cmap='Greens',
            xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'],
            ax=ax4, cbar=False)
ax4.set_title(f'FinBERT CM\nAcc: {finbert_acc:.2%}', fontweight='bold')

# 4. Training curves
ax5 = fig.add_subplot(3, 4, 5)
epochs_ran = range(1, len(history['train_loss']) + 1)
ax5.plot(epochs_ran, history['train_loss'], label='Train', color='#3498db')
ax5.plot(epochs_ran, history['val_loss'], label='Val', color='#e74c3c')
ax5.set_title('LSTM Loss Curve', fontweight='bold')
ax5.legend(fontsize=8)

ax6 = fig.add_subplot(3, 4, 6)
ax6.plot(epochs_ran, history['train_acc'], label='Train', color='#3498db')
ax6.plot(epochs_ran, history['val_acc'], label='Val', color='#e74c3c')
ax6.set_title('LSTM Accuracy Curve', fontweight='bold')
ax6.legend(fontsize=8)

# 5. LSTM Confusion Matrix
ax7 = fig.add_subplot(3, 4, 7)
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'],
            ax=ax7, cbar=False)
ax7.set_title(f'LSTM CM\nAcc: {lstm_acc:.2%}', fontweight='bold')

# 6. Final comparison
ax8 = fig.add_subplot(3, 4, 8)
ax8.bar(['VADER', 'FinBERT', 'LSTM'],
        [vader_acc, finbert_acc, lstm_acc],
        color=['#3498db', '#2ecc71', '#9b59b6'], edgecolor='black')
ax8.set_title('Model Accuracy Comparison', fontweight='bold')
ax8.set_ylim(0, 1)
for i, acc in enumerate([vader_acc, finbert_acc, lstm_acc]):
    ax8.text(i, acc + 0.01, f'{acc:.2%}', ha='center', fontsize=9, fontweight='bold')

# 7. Feature importance
ax9 = fig.add_subplot(3, 2, 5)
sorted_imp = importance_df.sort_values('Importance', ascending=True)
ax9.barh(sorted_imp['Feature'], sorted_imp['Importance'],
         color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(sorted_imp))))
ax9.set_title('Feature Importance (LSTM Gradients)', fontweight='bold')

# 8. Key findings text
ax10 = fig.add_subplot(3, 2, 6)
ax10.axis('off')
findings = [
    "KEY FINDINGS",
    "",
    f"Dataset: 4,846 financial headlines",
    f"  Neutral: {(df['sentiment']=='neutral').sum()} ({(df['sentiment']=='neutral').mean():.0%})",
    f"  Positive: {(df['sentiment']=='positive').sum()} ({(df['sentiment']=='positive').mean():.0%})",
    f"  Negative: {(df['sentiment']=='negative').sum()} ({(df['sentiment']=='negative').mean():.0%})",
    "",
    f"VADER Accuracy:   {vader_acc:.2%}",
    f"FinBERT Accuracy: {finbert_acc:.2%}",
    f"LSTM Accuracy:    {lstm_acc:.2%}",
    "",
    "✅ FinBERT outperforms VADER on",
    "   financial domain text",
    "✅ LSTM combines both for best",
    "   multi-class performance",
    "✅ VADER compound score is the",
    "   single most predictive feature",
]
ax10.text(0.05, 0.95, '\n'.join(findings), transform=ax10.transAxes,
          verticalalignment='top', fontsize=11, fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Summary dashboard saved!")

In [ ]:
# ── Save Model ────────────────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'label_encoder': label_encoder,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'input_size': INPUT_SIZE,
    'num_classes': NUM_CLASSES,
    'final_val_acc': max(history['val_acc']),
    'test_acc': lstm_acc,
}, 'financial_sentiment_lstm.pt')

print("✅ Model saved to 'financial_sentiment_lstm.pt'")
print(f"\n📌 Final Summary:")
print(f"   VADER Accuracy   : {vader_acc:.4f}")
print(f"   FinBERT Accuracy : {finbert_acc:.4f}")
print(f"   LSTM Accuracy    : {lstm_acc:.4f}")
print(f"\nAll charts saved as PNG files in working directory.")